# Distribution of Vulnerability Scores

This notebook calculates general statistics (mean, media, variance etc.) for mean vulnerability as well as for all individual variables of the baseline indicator. It subsequently generates boxplots describing the distribution of vulnerability scores for all leave-one-out scenarios as well as for the baseliene indicators. It creates:

- csv with general statistics for baseline indicator score distributions
- small-multiple plot with boxblots for vulnerability scores for all leave-one-out scenarios and baseline indicator

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `vulnerability_mean.tif`
- `vulnerability_mean__*.tif` (all ten leave-one-out scenarios)

In [ ]:
# Configuration (edit these paths if needed)
#Inputs
VULN = 'vulnerability_indicator\\vulnerability_mean.tif'
BASE = 'vulnerability_indicator\\scenarios_missing_variable\\scenarios_one_missing'

#Output
OUT_PNG= "vulnerability_boxplots_small_multiples.png"

In [ ]:
#import packages
import os
import rasterio
import numpy as np
import pandas as pd
from scipy import stats
import re
import math
import rasterio
import matplotlib.pyplot as plt

In [ ]:
# Save outputs

base_dir = BASE

filenames = [
    "vulnerability_mean__S_all__A_omit_edi_5000m.tif",
    "vulnerability_mean__S_omit_kba_5000m__A_all.tif",
    "vulnerability_mean__S_omit_landmark_5000m__A_all.tif",
    "vulnerability_mean__S_omit_poverty_5000m__A_all.tif",
    "vulnerability_mean__S_omit_water_risk_5000m__A_all.tif",
    "vulnerability_mean__S_omit_wdpa_5000m__A_all.tif",
    "vulnerability_mean__S_all__A_omit_conflict_5000m.tif",
    "vulnerability_mean__S_all__A_omit_landrights_5000m.tif",
    "vulnerability_mean__S_all__A_omit_rule_of_law_5000m.tif",
    "vulnerability_mean__S_omit_bii_5000m__A_all.tif",
    "vulnerability_mean__S_all__A_all.tif",
]

def extract_scenario(filename):
    if "omit_" in filename:
        return filename.split("omit_")[1].replace("_5000m.tif", "")
    return "all"

results = []

for fname in filenames:
    vuln_path = os.path.join(base_dir, fname)


    with rasterio.open(vuln_path) as src:
        arr = src.read(1)
        nodata = src.nodata

    # Mask NoData
    if nodata is not None:
        values = arr[arr != nodata]
    else:
        values = arr.ravel()

    values = values[np.isfinite(values)].astype(np.float64)

    # Statistics
    mean_val = np.mean(values)
    median_val = np.median(values)
    variance_val = np.var(values)
    std_val = np.sqrt(variance_val)
    skewness_val = stats.skew(values)
    kurtosis_val = stats.kurtosis(values)  # excess kurtosis
    p90_val = np.percentile(values, 90)
    mode_val = stats.mode(values, keepdims=False).mode
    max_val = np.max(values)


    results.append({
        "scenario": extract_scenario(fname),
        "mean": mean_val,
        "median": median_val,
        "mode": mode_val,
        "variance": variance_val,
        "std": std_val,
        "skewness": skewness_val,
        "excess_kurtosis": kurtosis_val,
        "p90": p90_val,
        "max": max_val,
    })

# Create table
df = pd.DataFrame(results)

# Round for readability
df = df.round(4)

# Put full model first
df["is_all"] = (df["scenario"] == "all").astype(int)
df = df.sort_values(["is_all", "scenario"], ascending=[False, True]).drop(columns="is_all")

print("\nVulnerability summary statistics")
print(df.to_string(index=False))

# Save
out_csv = os.path.join("vulnerability_summary_statistics.csv")
df.to_csv(out_csv, index=False)
print(f"\nSaved to {out_csv}")


In [ ]:
# Base directory containing all scenario rasters
base_dir = BASE

# List of scenario raster filenames (leave-one-out scenarios + baseline)
filenames = [
    "vulnerability_mean__S_all__A_omit_edi_5000m.tif",
    "vulnerability_mean__S_omit_kba_5000m__A_all.tif",
    "vulnerability_mean__S_omit_landmark_5000m__A_all.tif",
    "vulnerability_mean__S_omit_poverty_5000m__A_all.tif",
    "vulnerability_mean__S_omit_water_risk_5000m__A_all.tif",
    "vulnerability_mean__S_omit_wdpa_5000m__A_all.tif",
    "vulnerability_mean__S_all__A_omit_conflict_5000m.tif",
    "vulnerability_mean__S_all__A_omit_landrights_5000m.tif",
    "vulnerability_mean__S_all__A_omit_rule_of_law_5000m.tif",
    "vulnerability_mean__S_omit_bii_5000m__A_all.tif",
    "vulnerability_mean__S_all__A_all.tif",
]

# Output file path for the figure
out_png = OUT_PNG

# Mapping short variable names to readable labels
name = {
    "edi": "Environmental Democracy",
    "conflict": "Conflict",
    "landrights": "Landrights",
    "rule_of_law": "Rule of Law",
    "bii": "Biodiversity Intactness",
    "kba": "Key Biodiversity Areas",
    "landmark": "IPLC Lands",
    "poverty": "Poverty",
    "water_risk": "Water Risk",
    "wdpa": "Protected Areas",
}

# Extract which variable was omitted from the filename
def extract_scenario(filename):
    m = re.search(r"omit_(.+?)_5000m", filename)
    if m:
        return m.group(1)
    return "all"  # baseline scenario

# Load raster values
scenario_values = []

for fname in filenames:
    path = os.path.join(base_dir, fname)

    # read raster
    with rasterio.open(path) as src:
        arr = src.read(1)
        nodata = src.nodata

    # remove nodata values
    if nodata is not None:
        values = arr[arr != nodata]
    else:
        values = arr.ravel()

    # remove NaN / infinite values
    values = values[np.isfinite(values)].astype(np.float64)

    # store values and scenario label
    scenario_values.append({
        "scenario": extract_scenario(fname),
        "values": values,
    })

# convert list to dataframe
df_vals = pd.DataFrame(scenario_values)

# ensure baseline ("all variables") appears first
df_vals["is_all"] = (df_vals["scenario"] == "all").astype(int)
df_vals = df_vals.sort_values(["is_all", "scenario"], ascending=[False, True]).drop(columns="is_all")

# Plot small-multiple boxplots
n = len(df_vals)
ncols = 3
nrows = math.ceil(n / ncols)

# create subplot grid
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(5.5 * ncols, 3.5 * nrows),
    sharey=True,
    dpi=200,
)

axes = np.array(axes).reshape(-1)

# loop through scenarios and create boxplots
for ax, (_, row) in zip(axes, df_vals.iterrows()):
    scen = row["scenario"]
    values = row["values"]

    # draw boxplot
    bp = ax.boxplot(
        values,
        vert=True,
        widths=0.5,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(facecolor="#4C72B0", alpha=0.8),
        medianprops=dict(color="black", linewidth=2),
        whiskerprops=dict(color="#444444"),
        capprops=dict(color="#444444"),
    )

    # compute summary statistics
    mean_val = np.mean(values)
    q1, median, q3 = np.percentile(values, [25, 50, 75])
    p90 = np.percentile(values, 90)
    whisker_low = bp["whiskers"][0].get_ydata()[1]
    whisker_high = bp["whiskers"][1].get_ydata()[1]

    # mark mean and 90th percentile
    ax.scatter(1, mean_val, color="red", zorder=3, s=30)
    ax.scatter(1, p90, color="black", zorder=3, s=20)

    # helper function to label statistics
    def label(y, text, dy=0.0, dx=0.03):
        ax.text(
            1 + dx, y + dy,
            f"{text:.3f}",
            va="center",
            ha="left",
            fontsize=10,
            color="#222222",
        )

    # annotate values
    label(mean_val, mean_val, dy=0.015)
    label(whisker_low, whisker_low, dy=-0.025)
    label(whisker_high, whisker_high, dy=0.025)
    label(p90, p90, dy=0.03)

    # scenario title
    if scen == "all":
        title = "All variables"
    else:
        title = f"Variable left out: {name.get(scen, scen)}"

    ax.set_title(title, fontsize=11, fontweight="semibold")

    # plot styling
    ax.set_xticks([])
    ax.grid(True, axis="y", alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # legend markers
    ax.scatter([], [], color="red", s=30, label="Mean")
    ax.scatter([], [], color="black", s=20, label="90th percentile")
    ax.legend(frameon=False, fontsize=9, loc="upper left")

# turn off unused subplot panels
for ax in axes[len(df_vals):]:
    ax.axis("off")

# figure title
fig.suptitle(
    "Distribution of vulnerability scores\nBoxplots with annotated statistics",
    fontsize=16,
    fontweight="semibold",
    y=0.98,
)

# add y-axis label on first column of each row
for r in range(nrows):
    axes[r * ncols].set_ylabel("Vulnerability score")

plt.tight_layout(rect=[0, 0, 1, 0.95])

# save and display figure
fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)